## AuxTel AzEl offsets - 15-Apr-21

In this notebook, investigate az-el offsets from 11-Mar-21

In [2]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.time import Time, TimeDelta
from lsst_efd_client import EfdClient
from lsst.daf.persistence import Butler


In [3]:
REPO_DIR = '/project/shared/auxTel'
butler = Butler(REPO_DIR)
dayObs = '2021-03-11'
firstExpId = 2021031100346
lastExpId = 2021031100424

<ipython-input-3-0ef545bbea13>:2: FutureWarning: Gen2 Butler has been deprecated (Butler). It will be removed sometime after v23.0 but no earlier than the end of 2021.
  butler = Butler(REPO_DIR)
<ipython-input-3-0ef545bbea13>:2: FutureWarning: Gen2 Butler has been deprecated (LatissMapper). It will be removed sometime after v23.0 but no earlier than the end of 2021.
  butler = Butler(REPO_DIR)


In [66]:
# This queries the header information in the butler metadata
# This is a limited amount of information, but returns very quickly
visits = butler.queryMetadata('raw', ['expId', 'EXPTIME', 'imagetype', 'object', 'filter', 'DATE'],\
                              detector=0, dayObs=dayObs)
myVisits = []
for visit in visits:
    if (int(visit[0]) >= firstExpId) and (int(visit[0]) <= lastExpId):
        myVisits.append(visit)
        print(visit)

(2021031100346, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:25.819')
(2021031100347, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:38.545')
(2021031100348, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:52.002')
(2021031100349, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:05.546')
(2021031100350, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:19.047')
(2021031100351, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:32.595')
(2021031100352, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:45.298')
(2021031100353, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:58.350')
(2021031100354, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:12.509')
(2021031100355, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:26.707')
(2021031100356, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:40.856')
(2021031100357, 5.0, 'SKYEXP', 'HD 125371', 'RG610~emp

In [5]:
# Get EFD client
client = EfdClient('summit_efd')

In [50]:
# These are for finding the timestamps of the offset events
backUp = 5 # seconds before first image to get initial offset
start = Time(myVisits[0][5],scale='tai') - TimeDelta(backUp, format='sec')
end = Time(myVisits[-1][5],scale='tai')
timestamp = f"time >= {start} AND time <= {end}"

In [51]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.command_offsetAzEl", ['*'],
                                          start, end)

In [71]:
offsets.columns

Index(['az', 'el', 'num', 'private_host', 'private_identity',
       'private_kafkaStamp', 'private_origin', 'private_rcvStamp',
       'private_revCode', 'private_seqNum', 'private_sndStamp'],
      dtype='object')

In [67]:
# Now append the applied offsets to the list of visits
fullVisits = []
for i, visit in enumerate(myVisits):
    if i == 0:
        startTime = Time(myVisits[i][5],scale='tai',precision=0) - TimeDelta(backUp, format='sec')
        startTime = startTime.isot
    else:
        startTime = Time(myVisits[i][5],scale='tai',precision=0).isot
    if i == len(myVisits) - 1:
        endTime = Time(myVisits[i][5],scale='tai',precision=0) + TimeDelta(backUp, format='sec')
    else:
        endTime = Time(myVisits[i+1][5],scale='tai',precision=0).isot
    try:
        offset = offsets.loc[startTime:endTime].values
        if len(offset) == 1:
            newList = list(visit)
            newList.append(offset[0][0])
            newList.append(offset[0][1])
            fullVisits.append(newList)
        else:
            continue
    except:
        continue


In [68]:
len(fullVisits)

74

In [70]:
# The last two values are the applied offsets in az and el in arcseconds
fullVisits[0]

[2021031100346,
 5.0,
 'SKYEXP',
 'HD 125371',
 'RG610~empty',
 '2021-03-12T04:09:25.819',
 24.0818437581706,
 -132.48108466195112]

In [82]:
# This way goes directly to the fits files without using the butler
import glob
files = glob.glob(REPO_DIR+'/_parent/raw/2021-03-11/*')
print(len(files))
hduList = pf.open(files[346])
hdr = hduList[0].header
for key in hdr.keys():
    print(key, hdr[key])

0


NameError: name 'pf' is not defined

In [80]:
REPO_DIR = '/project/shared/auxTel'
butler = Butler(REPO_DIR)

test = butler.get('raw', dataId={'expId': 2021031100346, 'detector': 0, 'dayObs': '2021-03-11'})

NoResults: No locations for get: datasetType:raw_amp dataId:DataId(initialdata={'expId': 2021031100346, 'detector': 0, 'dayObs': '2021-03-11', 'channel': 1}, tag=set())